In [1]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)

PyTorch: 2.14.0+cpu
torchvision: 0.29.0+cpu


In [2]:
from torchvision.models import mobilenet_v2, squeezenet1_1

mobile = mobilenet_v2(weights=None)
squeeze = squeezenet1_1(weights=None)

print("MobileNetV2 classifier:")
print(mobile.classifier)

print("SqueezeNet classifier:")
print(squeeze.classifier)

MobileNetV2 classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)
SqueezeNet classifier:
Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Conv2d(512, 1000, kernel_size=(1, 1), stride=(1, 1))
  (2): ReLU(inplace=True)
  (3): AdaptiveAvgPool2d(output_size=(1, 1))
)


In [3]:
print("MobileNetV2 feature blocks:", len(mobile.features))
print("SqueezeNet feature blocks:", len(squeeze.features))

MobileNetV2 feature blocks: 19
SqueezeNet feature blocks: 13


In [4]:
from edgecnn.models.pretrained import count_trainable_vs_total

trainable, total = count_trainable_vs_total(mobile)

print("Trainable:", trainable)
print("Total:", total)

assert trainable == total

SyntaxError: from __future__ imports must occur at the beginning of the file (backbones.py, line 41)

In [ ]:
from edgecnn.contracts.types import EvalResult
from edgecnn.evaluation.reporting import plot_confusion_matrix

sample = EvalResult(
    run_id="model_b__adam__seed42",
    model_name="model_b",
    accuracy=0.79,
    macro_precision=0.78,
    macro_recall=0.79,
    macro_f1=0.78,
    per_class={
        "Forest": {"precision": 0.8, "recall": 0.9, "f1": 0.85, "support": 50},
        "River": {"precision": 0.76, "recall": 0.68, "f1": 0.72, "support": 50},
    },
    confusion_matrix=[[45, 5], [16, 34]],
    class_names=["Forest", "River"],
)

figure = plot_confusion_matrix(sample, normalize=True, write=False)
assert figure is not None

In [ ]:
from torchvision.models import mobilenet_v2, squeezenet1_1

from edgecnn.models.pretrained.finetune import (
    apply_finetune_strategy,
    count_trainable_vs_total,
    replace_classifier,
)

mobile = replace_classifier(mobilenet_v2(weights=None), 10, dropout=0.2)
mobile = apply_finetune_strategy(
    mobile,
    "last_n_blocks",
    unfreeze_last_n=3,
)

squeeze = replace_classifier(squeezenet1_1(weights=None), 10, dropout=0.5)
squeeze = apply_finetune_strategy(
    squeeze,
    "last_n_blocks",
    unfreeze_last_n=2,
)

for model in (mobile, squeeze):
    trainable, total = count_trainable_vs_total(model)
    print(type(model).__name__, trainable, total)
    assert model.num_classes == 10
    assert 0 < trainable < total

MobileNetV2 1218890 2236682
SqueezeNet 391306 727626


In [2]:
import torch

from edgecnn.models.pretrained.backbones import (
    build_mobilenet_v2,
    build_squeezenet,
    param_groups,
)
from edgecnn.models.pretrained import count_trainable_vs_total

mobile = build_mobilenet_v2(
    num_classes=10,
    weights=None,
    torchvision_name="mobilenet_v2",
    finetune_strategy="last_n_blocks",
    unfreeze_last_n=3,
    replace_classifier=True,
    dropout=0.2,
    backbone_lr_scale=0.1,
    input_resolution=64,
)

squeeze = build_squeezenet(
    num_classes=10,
    weights=None,
    torchvision_name="squeezenet1_1",
    finetune_strategy="last_n_blocks",
    unfreeze_last_n=2,
    replace_classifier=True,
    dropout=0.5,
    backbone_lr_scale=0.1,
    input_resolution=64,
)

for model in (mobile, squeeze):
    model.eval()
    with torch.no_grad():
        logits = model(torch.randn(1, 3, 64, 64))

    trainable, total = count_trainable_vs_total(model)
    print(type(model).__name__, logits.shape, trainable, total)

    assert logits.shape == (1, 10)
    assert logits.dtype == torch.float32
    assert model.num_classes == 10
    assert 0 < trainable < total
    assert not torch.allclose(
        logits.sum(dim=1),
        torch.ones(1),
        atol=1e-3,
    )

    groups = param_groups(
        model,
        base_lr=0.001,
        backbone_lr_scale=0.1,
    )
    learning_rates = {group["name"]: group["lr"] for group in groups}
    assert learning_rates["head"] == 0.001
    assert learning_rates["backbone"] == 0.0001

MobileNetV2 torch.Size([1, 10]) 1218890 2236682
SqueezeNet torch.Size([1, 10]) 391306 727626


In [3]:
from edgecnn.training.optimizers import build_optimizer

optimizer = build_optimizer(
    mobile,
    {
        "name": "adam",
        "lr": 0.001,
        "weight_decay": 0.0001,
        "betas": [0.9, 0.999],
        "backbone_lr_scale": 0.1,
    },
)

assert sorted(group["lr"] for group in optimizer.param_groups) == [
    0.0001,
    0.001,
]